# Grouped Query Attention (GQA): KV Cache Reduction and the Llama 2 Implementation

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/research_3)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q torch transformers peft bitsandbytes

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GroupedQueryAttention(nn.Module):
    """
    Grouped Query Attention as used in Llama 2 70B and Mistral 7B.
    """
    def __init__(self, d_model: int, num_q_heads: int, num_kv_heads: int):
        super().__init__()
        assert num_q_heads % num_kv_heads == 0, \
            f"num_q_heads ({num_q_heads}) must be divisible by num_kv_heads ({num_kv_heads})"

        self.num_q_heads = num_q_heads
        self.num_kv_heads = num_kv_heads
        self.num_groups = num_q_heads // num_kv_heads
        self.head_dim = d_model // num_q_heads

        # Query: full heads, KV: reduced heads
        self.q_proj = nn.Linear(d_model, num_q_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # Project to Q, K, V
        q = self.q_proj(x).view(B, T, self.num_q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # Expand KV heads to match Q heads: repeat each KV head num_groups times
        # (B, num_kv_heads, T, head_dim) → (B, num_q_heads, T, head_dim)
        k = k.repeat_interleave(self.num_groups, dim=1)
        v = v.repeat_interleave(self.num_groups, dim=1)

        # Scaled dot-product attention
        scale = math.sqrt(self.head_dim)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / scale
        attn_weights = F.softmax(attn_scores, dim=-1)

        # Weighted sum of values
        out = torch.matmul(attn_weights, v)  # (B, num_q_heads, T, head_dim)

        # Concatenate heads
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


# Test: MHA equivalent (num_kv_heads = num_q_heads)
mha = GroupedQueryAttention(d_model=512, num_q_heads=8, num_kv_heads=8)
# GQA with 4 groups
gqa_4 = GroupedQueryAttention(d_model=512, num_q_heads=8, num_kv_heads=2)
# MQA equivalent (num_kv_heads = 1)
mqa = GroupedQueryAttention(d_model=512, num_q_heads=8, num_kv_heads=1)

x = torch.randn(2, 16, 512)  # (batch=2, seq_len=16, d_model=512)

out_mha  = mha(x)
out_gqa  = gqa_4(x)
out_mqa  = mqa(x)

print(f"Output shape (all variants): {out_mha.shape}")
print(f"\nParameter counts:")
print(f"  MHA  (8 KV heads): {sum(p.numel() for p in mha.parameters()):,}")
print(f"  GQA  (2 KV heads): {sum(p.numel() for p in gqa_4.parameters()):,}")
print(f"  MQA  (1 KV head):  {sum(p.numel() for p in mqa.parameters()):,}")

In [ ]:
import torch

def kv_cache_memory_mb(
    num_layers: int,
    num_kv_heads: int,
    head_dim: int,
    seq_len: int,
    dtype_bytes: int = 2,  # float16
) -> float:
    """Memory in MB for the full KV cache."""
    # K cache + V cache
    bytes_total = 2 * num_layers * num_kv_heads * head_dim * seq_len * dtype_bytes
    return bytes_total / (1024 ** 2)

# Llama 2 7B configuration
config_7b = dict(num_layers=32, num_kv_heads=32, head_dim=128)  # MHA
# Llama 2 70B configuration (GQA with 8 KV heads)
config_70b = dict(num_layers=80, num_kv_heads=8, head_dim=128)  # GQA

seq_lengths = [2048, 4096, 8192, 32768]
print(f"{'Seq Len':>10} | {'Llama 2 7B (MHA)':>18} | {'Llama 2 70B (GQA)':>19}")
print("-" * 55)
for sl in seq_lengths:
    mem_7b  = kv_cache_memory_mb(**config_7b,  seq_len=sl)
    mem_70b = kv_cache_memory_mb(**config_70b, seq_len=sl)
    print(f"{sl:>10,} | {mem_7b:>16.1f} MB | {mem_70b:>17.1f} MB")

In [ ]:
import torch

# Simulate: 2 KV heads, 4 Q heads (num_groups = 2)
kv = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # shape (2, 2) = (num_kv_heads, head_dim)
print(f"KV before: {kv}")

# Expand: each KV head repeated 2 times
expanded = kv.repeat_interleave(2, dim=0)  # shape (4, 2)
print(f"KV after repeat_interleave(2): {expanded}")

In [ ]:
from transformers import AutoConfig

# Llama 2 70B
config = AutoConfig.from_pretrained("meta-llama/Llama-2-70b-hf")
print(f"Model: Llama 2 70B")
print(f"  num_attention_heads (Q): {config.num_attention_heads}")
print(f"  num_key_value_heads (KV): {config.num_key_value_heads}")
print(f"  GQA groups: {config.num_attention_heads // config.num_key_value_heads}")
print(f"  KV cache reduction: {config.num_attention_heads // config.num_key_value_heads}x")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

config = model.config
print(f"Q heads: {config.num_attention_heads}")
print(f"KV heads: {config.num_key_value_heads}")
print(f"GQA groups: {config.num_attention_heads // config.num_key_value_heads}")

# Inspect a single attention layer's projection sizes
attn_layer = model.model.layers[0].self_attn
print(f"\nLayer 0 projection shapes:")
print(f"  q_proj: {attn_layer.q_proj.weight.shape}")
print(f"  k_proj: {attn_layer.k_proj.weight.shape}")
print(f"  v_proj: {attn_layer.v_proj.weight.shape}")